In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

In [0]:
BRONZE_PATH = "/Volumes/databricks_wrkspce/default/banking_data/migration/bronze"
SILVER_PATH = "/Volumes/databricks_wrkspce/default/banking_data/migration/silver"

In [0]:
TABLE_CONFIG = {

    "customers": {
        "protected_columns": [
            "customer_id"
        ],
        "filter": "customer_id IS NOT NULL",
        "uppercase_columns": [
            "customer_segment"
        ],
        "deduplicate_by": [
            "customer_id"
        ]
    },

    "accounts": {
        "protected_columns": [
            "account_id",
            "customer_id",
            "branch_id"
        ],
        "filter": "account_id IS NOT NULL AND customer_id IS NOT NULL",
        "uppercase_columns": [],
        "deduplicate_by": [
            "account_id"
        ]
    },

    "branches": {
        "protected_columns": [
            "branch_id"
        ],
        "filter": "branch_id IS NOT NULL",
        "uppercase_columns": [
            "region"
        ],
        "deduplicate_by": [
            "branch_id"
        ]
    },

    "transactions": {
        "protected_columns": [
            "transaction_id",
            "account_id"
        ],
        "filter": "transaction_id IS NOT NULL AND account_id IS NOT NULL",
        "uppercase_columns": [
            "transaction_type",
            "transaction_status"
        ],
        "deduplicate_by": [
            "transaction_id"
        ]
    }
}

In [0]:
def read_bronze(table_name):

    path = f"{BRONZE_PATH}/{table_name}"

    return (
        spark.read
        .format("delta")
        .load(path)
    )

In [0]:
def apply_filters(df, filter_condition):

    if filter_condition:
        df = df.filter(filter_condition)

    return df

In [0]:
def apply_uppercase(df, columns):

    for column_name in columns:

        if column_name in df.columns:

            df = df.withColumn(
                column_name,
                F.upper(F.trim(F.col(column_name)))
            )

    return df

In [0]:
def apply_deduplication(df, key_columns):

    if key_columns:

        df = df.dropDuplicates(key_columns)

    return df

In [0]:
def write_silver(df, table_name):

    path = f"{SILVER_PATH}/{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(path)
    )

In [0]:
def report_metrics(
    table_name,
    bronze_count,
    filtered_count,
    silver_count
):

    print(f"\n{'=' * 50}")
    print(f"Dataset: {table_name}")
    print(f"{'=' * 50}")
    print(f"Bronze records      : {bronze_count}")
    print(f"After filtering     : {filtered_count}")
    print(f"After deduplication : {silver_count}")
    print(f"Records removed     : {bronze_count - silver_count}")

In [0]:
for table_name, config in TABLE_CONFIG.items():

    print(f"\nProcessing: {table_name}")

    # Read Bronze
    df = read_bronze(table_name)

    bronze_count = df.count()

    # Filter
    df = apply_filters(
        df,
        config["filter"]
    )

    filtered_count = df.count()

    # Standardize
    df = apply_uppercase(
        df,
        config["uppercase_columns"]
    )

    # Deduplicate
    df = apply_deduplication(
        df,
        config["deduplicate_by"]
    )

    silver_count = df.count()

    # Write Silver
    write_silver(
        df,
        table_name
    )

    # Report
    report_metrics(
        table_name,
        bronze_count,
        filtered_count,
        silver_count
    )

    print(f"{table_name} → Silver written successfully")


Processing: customers

Dataset: customers
Bronze records      : 10100
After filtering     : 10100
After deduplication : 10000
Records removed     : 100
customers → Silver written successfully

Processing: accounts

Dataset: accounts
Bronze records      : 20050
After filtering     : 20050
After deduplication : 20000
Records removed     : 50
accounts → Silver written successfully

Processing: branches

Dataset: branches
Bronze records      : 200
After filtering     : 200
After deduplication : 200
Records removed     : 0
branches → Silver written successfully

Processing: transactions

Dataset: transactions
Bronze records      : 501000
After filtering     : 501000
After deduplication : 500000
Records removed     : 1000
transactions → Silver written successfully
